# Custom De-ID Pipeline — 20 Test Cases
Tests various real-world phrasings for DOB, dates, patient names, doctors, ZIP codes, etc.
Results are exported to `DeID_Test_Results.xlsx`.

In [ ]:
import json, os, re, sys
from datetime import datetime, date as date_type

os.environ["JAVA_HOME"]             = "/opt/homebrew/opt/openjdk@11"
os.environ["PYSPARK_PYTHON"]        = sys.executable   # workers use venv Python
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

with open("spark_jsl.json") as f:
    license_keys = json.load(f)
locals().update(license_keys)
os.environ.update(license_keys)
print("License keys loaded. Python:", sys.executable)

In [ ]:
import sparknlp, sparknlp_jsl
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.ml import Pipeline, PipelineModel
import pandas as pd

spark = sparknlp_jsl.start(license_keys["SECRET"])
print("Spark NLP:", sparknlp.version(), "| JSL:", sparknlp_jsl.version())

In [ ]:
import re
from datetime import datetime, date as date_type
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# ── Patient ID lookup ──────────────────────────────────────────────────────
PATIENT_ID_MAP = {
    "Daniel Foster"   : "PT-10042",
    "Laura Foster"    : "PT-10043",
    "John Smith"      : "PT-20017",
    "Emily Clarke"    : "PT-30001",
    "Robert Thompson" : "PT-30002",
    "Sarah Johnson"   : "PT-30003",
    "Michael Brown"   : "PT-30004",
    "Jessica Taylor"  : "PT-30005",
    "William Davis"   : "PT-30006",
    "Linda Martinez"  : "PT-30007",
    "James Wilson"    : "PT-30008",
    "Patricia Moore"  : "PT-30009",
    "Charles Anderson": "PT-30010",
    "Barbara Thomas"  : "PT-30011",
}

# ── Custom De-ID UDF — fully self-contained (no outer closures) ───────────
@F.udf(StringType())
def custom_deid_udf(text, begins, ends, results, meta_list):
    # All imports inside UDF to avoid PySpark closure/pickle issues
    import re
    from datetime import datetime, date as date_type

    _PATIENT_ID_MAP = {
        "Daniel Foster"   : "PT-10042",
        "Laura Foster"    : "PT-10043",
        "John Smith"      : "PT-20017",
        "Emily Clarke"    : "PT-30001",
        "Robert Thompson" : "PT-30002",
        "Sarah Johnson"   : "PT-30003",
        "Michael Brown"   : "PT-30004",
        "Jessica Taylor"  : "PT-30005",
        "William Davis"   : "PT-30006",
        "Linda Martinez"  : "PT-30007",
        "James Wilson"    : "PT-30008",
        "Patricia Moore"  : "PT-30009",
        "Charles Anderson": "PT-30010",
        "Barbara Thomas"  : "PT-30011",
    }

    _DATE_FORMATS = [
        "%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d", "%m-%d-%Y", "%d-%m-%Y",
        "%d.%m.%Y", "%m.%d.%Y", "%Y/%m/%d",
        "%B %d, %Y", "%b %d, %Y", "%d %B %Y", "%d %b %Y",
    ]

    def _parse_date(s):
        s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', s.strip())
        for fmt in _DATE_FORMATS:
            try:
                return datetime.strptime(s, fmt)
            except ValueError:
                pass
        return None

    def _get_patient_id(name):
        return _PATIENT_ID_MAP.get(name, f"PT-{abs(hash(name)) % 90000 + 10000}")

    def _date_to_month_year(s):
        dt = _parse_date(s)
        if dt:
            return dt.strftime("%B %Y")
        m = re.search(r'\b(19|20)\d{2}\b', s)
        return m.group(0) if m else "[DATE]"

    def _dob_to_age(s):
        dt = _parse_date(s)
        if dt:
            today = date_type.today()
            age = today.year - dt.year - ((today.month, today.day) < (dt.month, dt.day))
            if 0 <= age <= 120:
                return f"{age} years old"
        return "[DOB]"

    def _partial_zip(z):
        z = z.strip()
        return z[:3] + "X" * (len(z) - 3) if len(z) > 3 else z

    if not text or not results:
        return text

    chunks = []
    for i, chunk_text in enumerate(results):
        b      = begins[i]    if begins    else 0
        e      = ends[i]      if ends      else 0
        meta   = meta_list[i] if meta_list else {}
        entity = meta.get("entity", "") if meta else ""
        chunks.append((b, e, entity, chunk_text))

    chunks.sort(key=lambda x: x[0], reverse=True)
    result = text
    for begin, end, entity, chunk_text in chunks:
        if   entity == "DOCTOR":              continue
        elif entity == "PATIENT":             replacement = _get_patient_id(chunk_text)
        elif entity == "DATE":                replacement = _date_to_month_year(chunk_text)
        elif entity == "DOB":                 replacement = _dob_to_age(chunk_text)
        elif entity in ("ZIP", "ZIPCODE"):    replacement = _partial_zip(chunk_text)
        else:                                 replacement = f"[{entity}]"
        result = result[:begin] + replacement + result[end + 1:]
    return result

# ── Entity summary UDF — also self-contained ──────────────────────────────
@F.udf(StringType())
def entity_summary_udf(results, meta_list):
    if not results:
        return ""
    parts = []
    for i, chunk_text in enumerate(results):
        meta   = meta_list[i] if meta_list else {}
        entity = meta.get("entity", "?") if meta else "?"
        parts.append(f"{chunk_text} -> {entity}")
    return " | ".join(parts)

print("UDFs registered (self-contained).")

In [ ]:
# ── Build & fit pipeline (models cached from previous run) ────────────────
documentAssembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
splitter = (InternalDocumentSplitter().setInputCols("document").setOutputCol("sentence")
            .setSplitMode("recursive").setSplitPatterns([r"\s+|(?<=\G.{512})"])
            .setPatternsAreRegex(True).setChunkSize(512).setChunkOverlap(50)
            .setEnableSentenceIncrement(True))
tokenizer     = Tokenizer().setInputCols("sentence").setOutputCol("token")
tokenizer_doc = Tokenizer().setInputCols("document").setOutputCol("token_doc")

labels = ["DOCTOR","PATIENT","DATE_OF_BIRTH","DATE","CITY","STREET","STATE",
          "COUNTRY","PHONE","EMAIL","ZIP","USERNAME","ID","BIOID",
          "ORGANIZATION","MEDICAL_RECORD_NUMBER","SSN","AGE"]
zero_shot_ner = (PretrainedZeroShotNERChunker
    .pretrained("zeroshot_ner_deid_subentity_docwise_medium","en","clinical/models")
    .setInputCols("sentence").setOutputCol("ner_zero_shot")
    .setPredictionThreshold(0.7).setLabels(labels).setBatchSize(8))

zip_parser     = (ContextualParserModel.pretrained("zip_parser","en","clinical/models")
                  .setInputCols(["document","token_doc"]).setOutputCol("zip_chunks"))
dob_parser     = (ContextualParserModel.pretrained("date_of_birth_parser","en","clinical/models")
                  .setInputCols(["document","token_doc"]).setOutputCol("dob_chunks"))
email_matcher  = (RegexMatcherInternalModel.pretrained("email_matcher","en","clinical/models")
                  .setInputCols(["document"]).setOutputCol("email_chunks"))
country_matcher= (TextMatcherInternalModel.pretrained("country_matcher","en","clinical/models")
                  .setInputCols(["document","token_doc"]).setOutputCol("country_chunks")
                  .setMergeOverlapping(True))

chunk_merge_ner = (ChunkMergeModel().setInputCols("ner_zero_shot").setOutputCol("ner_merged")
    .setMergeOverlapping(True).setSelectionStrategy("DiverseLonger")
    .setResetSentenceIndices(True)
    .setReplaceDict({"DATE_OF_BIRTH":"DOB","MEDICAL_RECORD_NUMBER":"MEDICALRECORD"}))
chunk_merge_rules = (ChunkMergeModel()
    .setInputCols("zip_chunks","email_chunks","dob_chunks","country_chunks")
    .setOutputCol("rules_merged").setMergeOverlapping(True).setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"]).setSelectionStrategy("Sequential"))
chunk_merge_final = (ChunkMergeModel()
    .setInputCols("ner_merged","rules_merged").setOutputCol("ner_chunk")
    .setMergeOverlapping(True).setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"]).setSelectionStrategy("Sequential"))

pipeline = Pipeline(stages=[documentAssembler,splitter,tokenizer,tokenizer_doc,
    zero_shot_ner,chunk_merge_ner,zip_parser,dob_parser,email_matcher,country_matcher,
    chunk_merge_rules,chunk_merge_final])

pipeline_model = pipeline.fit(spark.createDataFrame([[""]],("text",)))
print("Pipeline ready.")

In [ ]:
# ── 20 Test Cases ─────────────────────────────────────────────────────────
test_cases = [
    {
        "id": 1,
        "name": "Standard DOB label",
        "text": "Patient: John Smith. DOB: 03/22/1985. Seen by Dr. Alice Wong on 04/10/2024. ZIP: 94404."
    },
    {
        "id": 2,
        "name": "Born on (written phrase)",
        "text": "Emily Clarke was born on April 5, 1990. She visited our clinic on January 20, 2024. Address: 12 Oak St, Boston, MA 02115."
    },
    {
        "id": 3,
        "name": "Date of Birth (full label)",
        "text": "Name: Robert Thompson. Date of Birth: 15-07-1978. Appointment date: 2024-03-01. Postcode: LS8 4HE."
    },
    {
        "id": 4,
        "name": "D.O.B. abbreviation with dots",
        "text": "Patient Sarah Johnson. D.O.B.: 22.11.1965. Dr. Patel reviewed on 10.02.2024. SSN: 444-55-8888."
    },
    {
        "id": 5,
        "name": "Birth date (alternate phrasing)",
        "text": "Michael Brown, birth date 07/04/1972, was referred to Dr. Green on 15th March 2024. Contact: michael.brown@email.com."
    },
    {
        "id": 6,
        "name": "ISO date format (YYYY-MM-DD)",
        "text": "Jessica Taylor (DOB: 1988-09-14) attended on 2024-04-22. Treated by Dr. Karen Hill. MRN: 5247840."
    },
    {
        "id": 7,
        "name": "Born (short keyword)",
        "text": "William Davis, born 1969-12-30, called our helpline (07785 441229) after his visit on 2024-01-15."
    },
    {
        "id": 8,
        "name": "Birthday label",
        "text": "Linda Martinez, Birthday: 08/19/1980. Last seen: 03/05/2024. Username: lmartinez80. ZIP: 10001."
    },
    {
        "id": 9,
        "name": "UK postcode + written date",
        "text": "James Wilson was born on 3rd February 1975. His appointment was on 12th April 2024. Postcode: SW1A 1AA. Seen by Dr. Emma Clarke."
    },
    {
        "id": 10,
        "name": "Multiple patients in one record",
        "text": "Primary patient: Patricia Moore (DOB 05/30/1962). Emergency contact: Charles Anderson, born 12/01/1960. Both seen by Dr. Liu on 04/18/2024."
    },
    {
        "id": 11,
        "name": "Service date as 'admission date'",
        "text": "Barbara Thomas, DOB: 14.09.1970. Admission date: 22.01.2024. Treating physician: Dr. Samuel Okafor. Hospital: St Luke's Medical Center."
    },
    {
        "id": 12,
        "name": "US ZIP+4 format",
        "text": "Patient Daniel Foster, DOB: 04/11/1972. Address: 27 Oakfield Road, Manchester, M13 9PL. Zip: 94404-1234. Phone: (212) 555-7890."
    },
    {
        "id": 13,
        "name": "Doctor name must NOT be masked",
        "text": "John Smith was seen by Dr. Richard Feynman and Dr. Marie Curie on 03/10/2024. DOB: 1990-06-15. Email: j.smith@nhs.uk."
    },
    {
        "id": 14,
        "name": "SSN + MRN + Username",
        "text": "Patient Emily Clarke. SSN: 123-45-6789. MRN: MF-88201. Username: eclarke90. DOB: July 4, 1990. Visit: 04/01/2024."
    },
    {
        "id": 15,
        "name": "Multiple dates in one record",
        "text": "Robert Thompson, born 15-07-1978. Diagnosed on 01/10/2022. Follow-up scheduled for 05/20/2024. Reviewed by Dr. Aisha Patel. ZIP: 33101."
    },
    {
        "id": 16,
        "name": "International address + country",
        "text": "Sarah Johnson (DOB: 22/11/1965) is a resident of 14 Rue de Rivoli, Paris, France. She was seen remotely on 02/14/2024. Email: s.johnson@gmail.com."
    },
    {
        "id": 17,
        "name": "'Date seen' phrasing for service date",
        "text": "Michael Brown (born 07/04/1972). Date seen: April 15, 2024. Referred by Dr. Henry Chang. Phone: 0161 882 3300. Postcode: M1 3HF."
    },
    {
        "id": 18,
        "name": "'Seen on' + abbreviated month",
        "text": "Jessica Taylor, DOB 14 Sep 1988. Seen on 22 Apr 2024 by Dr. Olivia Scott. Address: 9888 Genesee Ave, USA. ZIP: 92037."
    },
    {
        "id": 19,
        "name": "Complex full clinical note",
        "text": (
            "Patient: William Davis, born 30/12/1969. NHS No: 882 441 9930. "
            "Referred to St. Mary's Hospital on 15 Jan 2024 by Dr. Thomas Nguyen. "
            "Address: 45 King Street, Leeds, LS1 2AB. Phone: 07785 123456. "
            "Email: w.davis@outlook.com. SSN: 321-76-5432. Username: wdavis69."
        )
    },
    {
        "id": 20,
        "name": "'Record date' vs 'DOB' disambiguation",
        "text": (
            "Record date: 04/28/2026. Patient: Linda Martinez. "
            "Date of Birth: 19/08/1980. Seen by Dr. Fatima Al-Rashid. "
            "MRN: 647390883. ZIP: 90210. Email: linda.m@hospital.org."
        )
    },
]

print(f"Loaded {len(test_cases)} test cases.")

In [ ]:
# ── Entity summary UDF (avoids F.transform lambda serialization issues) ───
@F.udf(StringType())
def entity_summary_udf(results, meta_list):
    if not results:
        return ""
    parts = []
    for i, chunk_text in enumerate(results):
        meta   = meta_list[i] if meta_list else {}
        entity = meta.get("entity", "?") if meta else "?"
        parts.append(f"{chunk_text} → {entity}")
    return " | ".join(parts)

# ── Run all test cases through the pipeline ───────────────────────────────
texts = [tc["text"] for tc in test_cases]
ids   = [tc["id"]   for tc in test_cases]
names = [tc["name"] for tc in test_cases]

input_df = spark.createDataFrame(
    list(zip(ids, names, texts)),
    ["test_id", "test_name", "text"]
)

ner_out = pipeline_model.transform(input_df)

# Apply custom de-ID
result_df = ner_out.withColumn(
    "deid_text",
    custom_deid_udf(
        F.col("text"),
        F.col("ner_chunk.begin"),
        F.col("ner_chunk.end"),
        F.col("ner_chunk.result"),
        F.col("ner_chunk.metadata"),
    )
).withColumn(
    "entities_detected",
    entity_summary_udf(
        F.col("ner_chunk.result"),
        F.col("ner_chunk.metadata"),
    )
)

print("All test cases processed.")

In [ ]:
# ── Preview results ───────────────────────────────────────────────────────
pd.set_option("display.max_colwidth", 120)

results_pd = (
    result_df
    .select("test_id", "test_name", "text", "entities_detected", "deid_text")
    .orderBy("test_id")
    .toPandas()
)

results_pd.columns = ["#", "Test Case", "Original Text", "Entities Detected", "De-Identified Text"]
results_pd

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT_FILE = "DeID_Test_Results.xlsx"

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Test Results"

# ── Styles ─────────────────────────────────────────────────────────────────
header_fill   = PatternFill("solid", fgColor="1F4E79")   # dark blue
header_font   = Font(bold=True, color="FFFFFF", size=11)
alt_fill      = PatternFill("solid", fgColor="DEEAF1")   # light blue
entity_fill   = PatternFill("solid", fgColor="FFF2CC")   # light yellow
deid_fill     = PatternFill("solid", fgColor="E2EFDA")   # light green
wrap_align    = Alignment(wrap_text=True, vertical="top")
center_align  = Alignment(horizontal="center", vertical="top")
thin_border   = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin")
)

# ── Headers ────────────────────────────────────────────────────────────────
headers = ["#", "Test Case", "Original Text", "Entities Detected", "De-Identified Text"]
col_widths = [5, 30, 55, 55, 55]

for col_idx, (hdr, width) in enumerate(zip(headers, col_widths), start=1):
    cell = ws.cell(row=1, column=col_idx, value=hdr)
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border    = thin_border
    ws.column_dimensions[get_column_letter(col_idx)].width = width

ws.row_dimensions[1].height = 22

# ── Rows ───────────────────────────────────────────────────────────────────
for row_idx, row in results_pd.iterrows():
    excel_row = row_idx + 2
    is_alt = (row_idx % 2 == 0)

    values = [row["#"], row["Test Case"], row["Original Text"],
              row["Entities Detected"], row["De-Identified Text"]]

    for col_idx, value in enumerate(values, start=1):
        cell = ws.cell(row=excel_row, column=col_idx, value=str(value) if value else "")
        cell.alignment = center_align if col_idx == 1 else wrap_align
        cell.border    = thin_border

        if col_idx == 4:    cell.fill = entity_fill       # Entities col → yellow
        elif col_idx == 5:  cell.fill = deid_fill          # De-ID col   → green
        elif is_alt:        cell.fill = alt_fill           # Alternate rows → blue

    ws.row_dimensions[excel_row].height = 80

# ── Freeze header row ─────────────────────────────────────────────────────
ws.freeze_panes = "A2"

# ── Summary sheet ─────────────────────────────────────────────────────────
ws2 = wb.create_sheet("Summary")
ws2.column_dimensions["A"].width = 28
ws2.column_dimensions["B"].width = 45

summary_data = [
    ("Pipeline",             "Custom De-ID Pipeline"),
    ("NER Model",            "zeroshot_ner_deid_subentity_docwise_medium"),
    ("Rule-based models",    "zip_parser, date_of_birth_parser, email_matcher, country_matcher"),
    ("Total test cases",     str(len(results_pd))),
    ("", ""),
    ("PATIENT",   "→ Patient ID from lookup map"),
    ("DOCTOR",    "→ Kept as-is (not masked)"),
    ("DATE",      "→ Month + Year only (e.g. March 2024)"),
    ("DOB",       "→ Age (e.g. 53 years old)"),
    ("ZIP",       "→ First 3 chars + XX (e.g. 944XX)"),
    ("All others","→ [ENTITY_LABEL]"),
]

for r, (k, v) in enumerate(summary_data, start=1):
    ws2.cell(row=r, column=1, value=k).font = Font(bold=True)
    ws2.cell(row=r, column=2, value=v)

wb.save(OUTPUT_FILE)
print(f"\n✓ Excel saved: {OUTPUT_FILE}")
print(f"  Rows: {len(results_pd)} test cases + header")